In [29]:
!/root/llm/je/bin/python -m pip install --upgrade transformers==5.8.0

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 11.7 MB/s  0:00:0011.7 MB/s eta 0:00:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.8.1
    Uninstalling transformers-5.8.1:
      Successfully uninstalled transformers-5.8.1


In [31]:
!/root/llm/je/bin/python -c "import torch, transformers; print(torch.__version__, torch.version.cuda, torch.cuda.is_available()); print(transformers.__version__)"

2.7.1+cu118 11.8 True
5.8.0


In [1]:
!which python
!which python3
!python -c "import sys; print(sys.executable)"
!python3 -c "import sys; print(sys.executable)"
!echo $CONDA_PREFIX
!echo $PATH

/root/llm/je/bin/python
/root/llm/je/bin/python3
/root/llm/je/bin/python
/root/llm/je/bin/python3
/root/anaconda3
/root/llm/je/bin:/usr/local/cuda-12.4/bin:/usr/local/cuda-12.4/bin:/root/.vscode-server/data/User/globalStorage/github.copilot-chat/debugCommand:/root/.vscode-server/data/User/globalStorage/github.copilot-chat/copilotCli:/root/.vscode-server/cli/servers/Stable-0958016b2af9f09bb4257e0df4a95e2f90590f9f/server/bin/remote-cli:/root/.nvm/versions/node/v22.22.2/bin:/usr/local/cuda-12.4/bin:/root/anaconda3/bin:/root/anaconda3/condabin:/usr/local/cuda-11.8/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin


In [2]:
import sys
import numpy as np
import pandas as pd

print("PYTHON:", sys.executable)
print("numpy:", np.__version__, np.__file__)
print("pandas:", pd.__version__, pd.__file__)

PYTHON: /root/llm/je/bin/python
numpy: 2.2.6 /root/llm/je/lib/python3.10/site-packages/numpy/__init__.py
pandas: 2.3.3 /root/llm/je/lib/python3.10/site-packages/pandas/__init__.py


In [32]:
from pathlib import Path
import os
import sys
import subprocess
import json
import pandas as pd
from datetime import datetime

import torch

In [4]:
path = "/root/llm/JOILang-Server"

In [41]:
from pathlib import Path
import os
import sys
import subprocess
import json
from datetime import datetime

import pandas as pd
import torch


# =============================================================================
# 0. Kernel / Python 환경 확인
# =============================================================================
print("=" * 100)
print("0. Kernel / Python 환경 확인")
print("KERNEL PYTHON:", sys.executable)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("torch cuda runtime:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "CUDA is not available in the current Jupyter kernel. "
        "Kernel이 /root/llm/je/bin/python인지 확인하세요."
    )

# resolve() 사용 금지: /root/llm/je/bin/python이 anaconda 원본으로 풀릴 수 있음
EXPECTED_PYTHON = os.path.abspath("/root/llm/je/bin/python")
CURRENT_PYTHON = os.path.abspath(sys.executable)

if CURRENT_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Wrong Jupyter kernel Python.\n"
        f"Expected: {EXPECTED_PYTHON}\n"
        f"Current : {CURRENT_PYTHON}\n"
        f"Jupyter에서 Kernel → Change Kernel → Python (/root/llm/je)로 바꾸세요."
    )


# =============================================================================
# 1. Repository / Script path 설정
# =============================================================================
print("=" * 100)
print("1. Repository / Script path 설정")
REPO = Path(path).absolute()
VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"
LOCAL_MODEL_BASE = REPO / "local_models"

assert REPO.exists(), REPO
assert SCRIPT.exists(), SCRIPT

print("REPO:", REPO)
print("VERSION_DIR:", VERSION_DIR)
print("SCRIPT:", SCRIPT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)
print("LOCAL_MODEL_BASE exists:", LOCAL_MODEL_BASE.exists())


# =============================================================================
# 2. JOILang local model / worker 환경변수 설정
# =============================================================================
print("=" * 100)
print("2. JOILang local model / worker 환경변수 설정")
# 이전 실행에서 남아 있을 수 있는 충돌 변수 제거
for key in [
    "JOI_V15_LOCAL_MODEL_NAME",
    "JOI_V14_LOCAL_MODEL_NAME",
    "JOI_V14_WORKER_PYTHON",
    "JOI_V15_PERSISTENT_WORKER",   # 중요: persistent worker를 끄지 않기 위해 제거
]:
    os.environ.pop(key, None)

os.environ["PYTHONUNBUFFERED"] = "1"

# persistent worker는 유지하되, local model 위치만 지정
os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"

# worker도 현재 Jupyter kernel python과 동일하게 고정
os.environ["JOI_V15_WORKER_PYTHON"] = sys.executable

# debug
os.environ["JOI_V15_DEBUG_WORKER"] = "1"
os.environ["JOI_V15_DEBUG_LOG"] = "/tmp/joi_v15_worker_debug.log"

print("JOI_V15_WORKER_PYTHON:", os.environ.get("JOI_V15_WORKER_PYTHON"))
print("JOI_V15_LOCAL_MODEL_BASE_DIR:", os.environ.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
print("JOI_V15_LOCAL_DEVICE:", os.environ.get("JOI_V15_LOCAL_DEVICE"))
print("JOI_V15_LOCAL_FILES_ONLY:", os.environ.get("JOI_V15_LOCAL_FILES_ONLY"))
print("JOI_V15_PERSISTENT_WORKER:", os.environ.get("JOI_V15_PERSISTENT_WORKER"))
print("JOI_V15_DEBUG_LOG:", os.environ.get("JOI_V15_DEBUG_LOG"))


# =============================================================================
# 3. subprocess에서도 CUDA가 정상인지 확인
# =============================================================================
print("3. subprocess에서도 CUDA가 정상인지 확인")
subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import sys; "
            "print('subprocess python:', sys.executable); "
            "import torch; "
            "print('subprocess torch:', torch.__version__); "
            "print('subprocess cuda runtime:', torch.version.cuda); "
            "print('subprocess cuda available:', torch.cuda.is_available()); "
            "assert torch.cuda.is_available(), 'CUDA is not available in subprocess'; "
            "print('subprocess gpu:', torch.cuda.get_device_name(0))"
        ),
    ],
    check=True,
)

print("=" * 100)
print("Environment setup complete.")

0. Kernel / Python 환경 확인
KERNEL PYTHON: /root/llm/je/bin/python
pandas: 2.3.3
torch: 2.7.1+cu118
torch cuda runtime: 11.8
cuda available: True
GPU: NVIDIA A100 80GB PCIe
1. Repository / Script path 설정
REPO: /root/llm/JOILang-Server
VERSION_DIR: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413
SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py
RESULTS_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results
LOCAL_MODEL_BASE: /root/llm/JOILang-Server/local_models
LOCAL_MODEL_BASE exists: True
2. JOILang local model / worker 환경변수 설정
JOI_V15_WORKER_PYTHON: /root/llm/je/bin/python
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_DEBUG_LOG: /tmp/joi_v15_worker_debug.log
3. subprocess에서도 CUDA가 정상인지 확인
subprocess python: /root/llm/je/bin/python
subprocess torch: 2.7.1+cu118
subprocess cuda runtime: 11.8
subproc

In [42]:
from pathlib import Path
import os
import sys
import subprocess
from datetime import datetime

def run_ga_smoke_pair(
    label: str,
    model_key: str,
    target_detpass: float,
    use_cloud_advisor: bool,
    categories=(1, 2),
    limit_per_category=2,
    population=2,
    gens=3,
    sample_size=4,
    validation_size=4,
    timeout_sec=600,
    advisor_trigger_mode: str = "always",
    advisor_min_population_for_child: int = 4,
    advisor_force_child_quota: bool = True,
    use_mock_advisor: bool = False,
    launcher_python: str | None = None,
    worker_python: str | None = None,
    force_worker_mode: bool = False,
):
    if launcher_python is None:
        launcher_python = sys.executable
    if worker_python is None:
        worker_python = launcher_python

    # resolve() 금지: /root/llm/je/bin/python이 anaconda 원본으로 풀릴 수 있음
    launcher_python = os.path.abspath(os.path.expanduser(launcher_python))
    worker_python = os.path.abspath(os.path.expanduser(worker_python))

    if not Path(launcher_python).exists():
        raise FileNotFoundError(f"launcher_python does not exist: {launcher_python}")
    if not Path(worker_python).exists():
        raise FileNotFoundError(f"worker_python does not exist: {worker_python}")

    if use_cloud_advisor and not use_mock_advisor:
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError(
                "OPENAI_API_KEY is not set. "
                "Set it with getpass before running real cloud-advisor mode."
            )

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    cat_text = "".join(str(c) for c in categories)

    mode = "cloud_advisor" if use_cloud_advisor else "cloudless"
    mock_tag = "_mock" if use_mock_advisor and use_cloud_advisor else ""
    worker_tag = "_forcedworker" if force_worker_mode else "_persistent_default"
    safe_model_key = model_key.replace("/", "_").replace(":", "_")

    out_dir = (
        RESULTS_ROOT
        / f"ga_smoke_{mode}{mock_tag}{worker_tag}_cat{cat_text}_lpc{limit_per_category}"
          f"_pop{population}_gens{gens}_{safe_model_key}_{ts}"
        / "ga_output"
    )

    cmd = [
        launcher_python,
        "-u",
        str(SCRIPT),
        "--profile", "version0_15",
        "--model-key", model_key,
        "--target-detpass", str(target_detpass),

        "--population", str(population),
        "--gens", str(gens),
        "--min-generations", str(gens),
        "--max-generations", str(gens),

        "--sample-size", str(sample_size),
        "--validation-size", str(validation_size),
        "--cheap-eval-limit", "1",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--feedback-guided-mutation",

        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",

        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--plateau-window", "1",
        "--disruptive-max-attempts", "1",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",

        "--progress", "verbose",
        "--timeout-sec", str(timeout_sec),
        "--retries", "0",
        "--full-run",
        "--limit-per-category", str(limit_per_category),
        "--output-root", str(out_dir),
    ]

    # 기본은 False. True로 하면 version0_13/qwen_local_worker.py fallback 경로를 탈 수 있음.
    if force_worker_mode:
        cmd += ["--llm-mode", "worker"]

    for c in categories:
        cmd += ["--category", str(c)]

    if use_cloud_advisor:
        cmd += [
            "--llm-mutation-advisor",
            "--advisor-model-key", "gpt41_mini",
            "--advisor-trigger-mode", advisor_trigger_mode,
            "--advisor-min-population-for-child", str(advisor_min_population_for_child),
        ]

        if advisor_force_child_quota:
            cmd += ["--advisor-force-child-quota"]

        if use_mock_advisor:
            cmd += ["--llm-mode", "mock"]
    else:
        cmd += ["--advisor-trigger-mode", "off"]

    run_env = os.environ.copy()

    # A6000 성공 조건과 맞추기 위해 강제 local binding / persistent-off 제거
    # 충돌 변수만 제거
    for k in [
        "JOI_V15_LOCAL_MODEL_NAME",
        "JOI_V14_LOCAL_MODEL_NAME",
        "JOI_V14_WORKER_PYTHON",
        "JOI_V15_PERSISTENT_WORKER",  # 중요: false로 남아 있으면 안 됨
    ]:
        run_env.pop(k, None)

    run_env["PYTHONUNBUFFERED"] = "1"
    run_env["JOI_V15_WORKER_PYTHON"] = worker_python
    
    # local model은 명시
    run_env["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(REPO / "local_models")
    run_env["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
    run_env["JOI_V15_LOCAL_FILES_ONLY"] = "true"
    
    # debug log는 run별로 분리
    debug_log = f"/tmp/joi_v15_worker_debug_{safe_model_key}_{ts}.log"
    run_env["JOI_V15_DEBUG_WORKER"] = "1"
    run_env["JOI_V15_DEBUG_LOG"] = debug_log

    print("=" * 100)
    print(f"RUN: {label} / {model_key} / {mode}{mock_tag}{worker_tag}")
    print("OUTPUT:", out_dir)
    print("KERNEL_PYTHON:", sys.executable)
    print("LAUNCHER_PYTHON:", launcher_python)
    print("WORKER_PYTHON:", run_env.get("JOI_V15_WORKER_PYTHON"))
    print("FORCE_WORKER_MODE:", force_worker_mode)
    print("JOI_V15_LOCAL_MODEL_BASE_DIR:", run_env.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
    print("JOI_V15_LOCAL_DEVICE:", run_env.get("JOI_V15_LOCAL_DEVICE"))
    print("JOI_V15_LOCAL_FILES_ONLY:", run_env.get("JOI_V15_LOCAL_FILES_ONLY"))
    print("JOI_V15_PERSISTENT_WORKER:", run_env.get("JOI_V15_PERSISTENT_WORKER"))
    print("JOI_V15_LOCAL_MODEL_NAME:", run_env.get("JOI_V15_LOCAL_MODEL_NAME"))
    print("DEBUG_LOG:", debug_log)
    print("COMMAND:")
    print(" ".join(cmd))
    print("=" * 100)

    proc = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()

    print("\nRETURN CODE:", rc)
    print("OUTPUT:", out_dir)
    print("DEBUG_LOG:", debug_log)

    if rc != 0:
        raise RuntimeError(
            f"{label} {mode}{mock_tag}{worker_tag} failed with return code {rc}"
        )

    return out_dir, debug_log

In [43]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_cloudless_cat1_persistent_localmodel",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1,),
    limit_per_category=1,
    population=2,
    gens=1,
    sample_size=1,
    validation_size=1,
    timeout_sec=300,
    launcher_python="/root/llm/je/bin/python",
    worker_python="/root/llm/je/bin/python",
    force_worker_mode=False,
)

RUN: smoke_cloudless_cat1_persistent_default / qwen25_coder_7b / cloudless_persistent_default
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_persistent_default_cat1_lpc1_pop2_gens1_qwen25_coder_7b_20260602_211248/ga_output
KERNEL_PYTHON: /root/llm/je/bin/python
LAUNCHER_PYTHON: /root/llm/je/bin/python
WORKER_PYTHON: /root/llm/je/bin/python
FORCE_WORKER_MODE: False
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_LOCAL_MODEL_NAME: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_211248.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_7b --target-detpass 100.0 --population 2 --gens 1 --min-generations 1 --max-generations 1 --sample-size 1 --validation-size 1 --cheap-eval-limit 1 --candid

#### debug log를 확인

In [44]:
from pathlib import Path

p = Path(debug_log)
print("debug_log:", p)
print("exists:", p.exists())

if p.exists():
    text = p.read_text(encoding="utf-8", errors="replace")
    print(text[-10000:])

debug_log: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_211248.log
exists: True

[2026-06-02T21:12:50] START persistent worker
worker_python=/root/llm/je/bin/python
worker_path=/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/utils/persistent_qwen_worker.py
cwd=/root/llm/JOILang-Server
JOI_V15_LOCAL_DEVICE=cuda:0
TRANSFORMERS_VERBOSITY=error
HF_HUB_DISABLE_PROGRESS_BARS=1
TOKENIZERS_PARALLELISM=false
PYTHONUNBUFFERED=1
PYTHONFAULTHANDLER=1
payload_summary={"keys": ["local_attn_implementation", "local_device", "local_dtype", "local_files_only", "local_hf_modules_cache", "local_load_in_4bit", "local_max_new_tokens", "local_model_name", "local_trust_remote_code", "messages", "model"], "model": "Qwen/Qwen2.5-Coder-7B-Instruct", "local_model_name": "/root/llm/JOILang-Server/local_models/qwen25_coder_7b", "local_device": "cuda:0", "local_dtype": "bf16", "local_files_only": true, "local_load_in_4bit": false, "local_max_new_tokens": 256, "message_lengths": [{"role": "system", "

# 1.다음 테스트 순서
## 1단계: category 1, sample 조금 확대

In [45]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_cloudless_cat1_sample2",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1,),
    limit_per_category=2,
    population=2,
    gens=1,
    sample_size=2,
    validation_size=2,
    timeout_sec=600,
    launcher_python="/root/llm/je/bin/python",
    worker_python="/root/llm/je/bin/python",
    force_worker_mode=False,
)

RUN: smoke_cloudless_cat1_sample2 / qwen25_coder_7b / cloudless_persistent_default
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_persistent_default_cat1_lpc2_pop2_gens1_qwen25_coder_7b_20260602_211619/ga_output
KERNEL_PYTHON: /root/llm/je/bin/python
LAUNCHER_PYTHON: /root/llm/je/bin/python
WORKER_PYTHON: /root/llm/je/bin/python
FORCE_WORKER_MODE: False
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_LOCAL_MODEL_NAME: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_211619.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_7b --target-detpass 100.0 --population 2 --gens 1 --min-generations 1 --max-generations 1 --sample-size 2 --validation-size 2 --cheap-eval-limit 1 --candidate-k 1 --r

## 2단계: category 1,2 smoke

In [46]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_cloudless_cat12",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1, 2),
    limit_per_category=2,
    population=2,
    gens=2,
    sample_size=4,
    validation_size=4,
    timeout_sec=900,
    launcher_python="/root/llm/je/bin/python",
    worker_python="/root/llm/je/bin/python",
    force_worker_mode=False,
)

RUN: smoke_cloudless_cat12 / qwen25_coder_7b / cloudless_persistent_default
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_persistent_default_cat12_lpc2_pop2_gens2_qwen25_coder_7b_20260602_212523/ga_output
KERNEL_PYTHON: /root/llm/je/bin/python
LAUNCHER_PYTHON: /root/llm/je/bin/python
WORKER_PYTHON: /root/llm/je/bin/python
FORCE_WORKER_MODE: False
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_LOCAL_MODEL_NAME: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_212523.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_7b --target-detpass 100.0 --population 2 --gens 2 --min-generations 2 --max-generations 2 --sample-size 4 --validation-size 4 --cheap-eval-limit 1 --candidate-k 1 --repair-

## 3단계: 기존 GA 설정에 가깝게 확대

In [47]:
out, debug_log = run_ga_smoke_pair(
    label="smoke_cloudless_cat12_pop5_gens3",
    model_key="qwen25_coder_7b",
    target_detpass=100.0,
    use_cloud_advisor=False,
    categories=(1, 2),
    limit_per_category=3,
    population=5,
    gens=3,
    sample_size=6,
    validation_size=6,
    timeout_sec=1800,
    launcher_python="/root/llm/je/bin/python",
    worker_python="/root/llm/je/bin/python",
    force_worker_mode=False,
)

RUN: smoke_cloudless_cat12_pop5_gens3 / qwen25_coder_7b / cloudless_persistent_default
OUTPUT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_persistent_default_cat12_lpc3_pop5_gens3_qwen25_coder_7b_20260602_213202/ga_output
KERNEL_PYTHON: /root/llm/je/bin/python
LAUNCHER_PYTHON: /root/llm/je/bin/python
WORKER_PYTHON: /root/llm/je/bin/python
FORCE_WORKER_MODE: False
JOI_V15_LOCAL_MODEL_BASE_DIR: /root/llm/JOILang-Server/local_models
JOI_V15_LOCAL_DEVICE: cuda:0
JOI_V15_LOCAL_FILES_ONLY: true
JOI_V15_PERSISTENT_WORKER: None
JOI_V15_LOCAL_MODEL_NAME: None
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_213202.log
COMMAND:
/root/llm/je/bin/python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile version0_15 --model-key qwen25_coder_7b --target-detpass 100.0 --population 5 --gens 3 --min-generations 3 --max-generations 3 --sample-size 6 --validation-size 6 --cheap-eval-limit 1 --candidate-k 

KeyboardInterrupt: 

# 2. 기타 환경변수 통일 (with a6000 server)

In [20]:
from pathlib import Path
import json
import pandas as pd

out_dir = Path("/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/ga_smoke_cloudless_cat1_lpc1_pop2_gens1_qwen25_coder_7b_20260602_201258/ga_output")

print("exists:", out_dir.exists())
print("files:")
for p in sorted(out_dir.iterdir()):
    print("-", p.name)

exists: True
files:
- accepted_genomes
- advisor_mutation_proposals.jsonl
- advisor_mutation_summary.csv
- advisor_prompt_generation_001.txt
- advisor_response_generation_001.json
- best_genome.json
- best_genomes.json
- best_prompt_metadata.json
- candidates
- checkpoints
- cloudless_prompt_units.jsonl
- evaluations
- ga_block_diffs.jsonl
- ga_generation_progress.csv
- ga_generation_progress.jsonl
- ga_pareto_frontier.csv
- ga_pareto_frontier.jsonl
- ga_pareto_generation_summary.csv
- ga_pareto_generation_summary.jsonl
- ga_population_diagnostics.csv
- ga_population_diagnostics.jsonl
- ga_run_manifest.json
- ga_summary.json
- ga_topk_genomes.csv
- mutation_operator_credit.csv
- mutation_proposals.csv
- mutation_proposals.jsonl
- pareto_archive.csv
- pareto_archive.jsonl
- population_transitions.csv
- promotion_decisions.csv
- promotion_decisions.json
- stage_status
- structured_feedback.jsonl
- structured_feedback_summary.csv


In [21]:
summary_path = out_dir / "ga_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

summary

{'best_history': [{'generation': 1,
   'genome_id': 'gen-378892e9-ecc3-87ab-8b45-85023a0286cc',
   'fitness': 0.0,
   'avg_det_score': 0.0,
   'validation_avg_det_score': 0.0,
   'train_det_pass_rate': 0.0,
   'train_gt_exact_rate': 0.0,
   'validation_det_pass_rate': 0.0,
   'validation_gt_exact_rate': 0.0,
   'genome': {'id': 'gen-378892e9-ecc3-87ab-8b45-85023a0286cc',
    'seed': 626297831,
    'blocks': ['01', '02', '05', '06'],
    'params': {'model': 'Qwen/Qwen2.5-Coder-7B-Instruct',
     'temperature': 0.0,
     'few_shot_count': 3,
     'max_tokens': 1024,
     'candidate_strategies': ['minimal',
      'direct',
      'canonical_names_first',
      'compact_json']},
    'block_params': {'02': {'few_shot_count': 3,
      'micro_rules': ['For INTEGER and DOUBLE arguments, avoid quoted numeric literals.']},
     '05': {'repair_mode': 'conservative', 'few_shot_count': 3}},
    '_ga_metadata': {'parent_ids': [],
     'mutation_types': ['initial_random'],
     'crossover_used': False

In [22]:
pd.read_csv(out_dir / "ga_generation_progress.csv")

,profile,generation,model_key,genome_id,parent_ids,model,fitness,avg_det_score,det_pass_rate,det_variance,...,generation_phase,plateau_type,next_action,stop_candidate,stop_reason,unique_prompt_hash_count,pareto_archive_size,pareto_archive_delta,disruptive_attempt_count,advisor_triggered
0,version0_15,1,qwen25_coder_7b,gen-378892e9-ecc3-87ab-8b45-85023a0286cc,NaN,Qwen/Qwen2.5-Coder-7B-Instruct,0.0,0.0,0.0,0.0,...,FINAL_SELECTION,max_generation_reached,stop_and_finalize,True,max generations reached,2,2,2,0,False


In [24]:
pd.read_csv(out_dir / "ga_population_diagnostics.csv")

,generation,model_key,category,row_evaluations,avg_det_score,det_pass_count,det_pass_rate,failure_histogram
0,1,qwen25_coder_7b,1,2,0.0,0,0.0,"{""invalid_json"": 2}"


In [15]:
with open(out_dir / "structured_feedback.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line[:1000])
        if i >= 5:
            break

{"row_id": 1, "model_key": "qwen25_coder_7b", "genome_id": "gen-378892e9-ecc3-87ab-8b45-85023a0286cc", "generation": 1, "det_profile": "strict", "failure_type": "invalid_json", "failure_count": 1, "affected_block_family": "Output_Schema", "suggested_mutation_type": "strengthen_json_only_rule", "original_failure_reasons": "invalid_json", "prompt_block_id": "03", "timestamp": "2026-06-02T20:13:42"}

{"row_id": 1, "model_key": "qwen25_coder_7b", "genome_id": "gen-e6f98332-8830-5d32-df5c-e9fedd38fc7e", "generation": 1, "det_profile": "strict", "failure_type": "invalid_json", "failure_count": 1, "affected_block_family": "Output_Schema", "suggested_mutation_type": "strengthen_json_only_rule", "original_failure_reasons": "invalid_json", "prompt_block_id": "03", "timestamp": "2026-06-02T20:13:42"}



In [16]:
debug_log = Path("/tmp/joi_v15_worker_debug_qwen25_coder_7b_20260602_201258.log")

print("exists:", debug_log.exists())
if debug_log.exists():
    text = debug_log.read_text(encoding="utf-8", errors="replace")
    print(text[-5000:])

exists: True
", line 288, in __getattr__\n    raise AttributeError\nAttributeError\n"}
stdout_tail={"ok": false, "error": "", "error_type": "worker_crash", "oom_flag": false, "traceback": "Traceback (most recent call last):\n  File \"/root/llm/je/lib/python3.10/site-packages/transformers/tokenization_utils_base.py\", line 286, in __getattr__\n    return self.data[item]\nKeyError: 'shape'\n\nDuring handling of the above exception, another exception occurred:\n\nTraceback (most recent call last):\n  File \"/root/llm/JOILang-Server/gpt_mg/version0_13/qwen_local_worker.py\", line 296, in main\n    generated = model.generate(\n  File \"/root/llm/je/lib/python3.10/site-packages/torch/utils/_contextlib.py\", line 116, in decorate_context\n    return func(*args, **kwargs)\n  File \"/root/llm/je/lib/python3.10/site-packages/transformers/generation/utils.py\", line 2415, in generate\n    batch_size = inputs_tensor.shape[0]\n  File \"/root/llm/je/lib/python3.10/site-packages/transformers/tokeniza

# 3. 본격 테스트
## “환경 설정 → 실행 wrapper → 6개 run 실행 → 요약/Delta 생성”
A6000 SetA = 중간 규모, 안정성/비용/시간 균형

A100  SetB = 원래 full cloudless 조건에 가까운 본 실험 규모



## Cell 1. 기본 경로 / 환경 / 서버 preset 설정


In [ ]:
from pathlib import Path
import os
import sys
import json
import time
import subprocess
import traceback
from datetime import datetime

import pandas as pd


# =============================================================================
# 0. Server preset 선택
# =============================================================================

# A6000 서버에서는 이 값 사용
# SERVER_PRESET = "A6000_SET_A"

# A100 서버에서는 위 줄을 주석 처리하고 아래 줄 사용
SERVER_PRESET = "A100_SET_B"


# =============================================================================
# 1. Repository / script path 설정
# =============================================================================

try:
    REPO = Path(path).absolute()
except NameError:
    REPO = Path.cwd().absolute()

VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
RESULTS_ROOT = VERSION_DIR / "results"
LOCAL_MODEL_BASE = REPO / "local_models"

assert REPO.exists(), REPO
assert SCRIPT.exists(), SCRIPT

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("SERVER_PRESET:", SERVER_PRESET)
print("REPO:", REPO)
print("SCRIPT:", SCRIPT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)
print("LOCAL_MODEL_BASE exists:", LOCAL_MODEL_BASE.exists())
print("PYTHON:", sys.executable)


# =============================================================================
# 2. 모델 목록
# =============================================================================

MODEL_LIST = [
    ("7B", "qwen25_coder_7b"),
    ("8B", "llama31_8b"),
    ("14B", "qwen25_coder_14b"),
]

RUN_MODES = [
    ("cloudless", False),
    ("cloud_advisor", True),
]


# =============================================================================
# 3. 서버별 실험 설정
# =============================================================================

# A6000: 중간 규모. 48GB급에서 14B까지 안정적으로 비교하기 위한 SetA.
SET_A_A6000 = dict(
    target_detpass=90,
    categories=range(1, 9),
    limit_per_category=2,
    sample_size=16,
    validation_size=16,
    population=4,
    gens=6,
    full_run=True,
    progress="verbose",
    timeout_sec=900,
    retries=0,
    idle_timeout_sec=2400,
    total_timeout_sec=24 * 3600,
)

# A100: 기존 cloudless full 설정에 가까운 본 비교 SetB.
SET_B_A100 = dict(
    target_detpass=90,
    categories=range(1, 9),
    limit_per_category=3,
    sample_size=24,
    validation_size=24,
    population=5,
    gens=10,
    full_run=True,
    progress="verbose",
    timeout_sec=1200,
    retries=0,
    idle_timeout_sec=3600,
    total_timeout_sec=40 * 3600,
)

if SERVER_PRESET == "A6000_SET_A":
    COMMON_GA_CONFIG = SET_A_A6000
elif SERVER_PRESET == "A100_SET_B":
    COMMON_GA_CONFIG = SET_B_A100
else:
    raise ValueError(f"Unknown SERVER_PRESET: {SERVER_PRESET}")

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
COMPARISON_ROOT = RESULTS_ROOT / f"fair_compare_{SERVER_PRESET}_{RUN_TAG}"
COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("COMMON_GA_CONFIG:", COMMON_GA_CONFIG)
print("COMPARISON_ROOT:", COMPARISON_ROOT)


# =============================================================================
# 4. 환경변수 설정
# =============================================================================

def setup_env_for_current_server():
    os.environ["PYTHONUNBUFFERED"] = "1"
    os.environ["JOI_V15_WORKER_PYTHON"] = sys.executable

    # 공통 충돌 변수 제거
    os.environ.pop("JOI_V15_LOCAL_MODEL_NAME", None)
    os.environ.pop("JOI_V14_LOCAL_MODEL_NAME", None)
    os.environ.pop("JOI_V14_WORKER_PYTHON", None)

    # 중요: persistent worker를 끄면 안 됨.
    # false가 남아 있으면 version0_13/qwen_local_worker.py fallback 가능.
    os.environ.pop("JOI_V15_PERSISTENT_WORKER", None)

    if SERVER_PRESET == "A100_SET_B":
        # A100에서는 offline local model 경로를 명확히 고정
        os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
        os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
        os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"

    elif SERVER_PRESET == "A6000_SET_A":
        # A6000에서 기존 성공 환경을 최대한 보존.
        # 이미 잘 도는 경우 LOCAL_MODEL_BASE_DIR을 강제하지 않음.
        # 필요할 때만 아래 3줄을 켜면 됨.
        # os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
        # os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"
        # os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"
        pass

    print("=" * 100)
    print("JOI_V15_WORKER_PYTHON:", os.environ.get("JOI_V15_WORKER_PYTHON"))
    print("JOI_V15_LOCAL_MODEL_BASE_DIR:", os.environ.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
    print("JOI_V15_LOCAL_DEVICE:", os.environ.get("JOI_V15_LOCAL_DEVICE"))
    print("JOI_V15_LOCAL_FILES_ONLY:", os.environ.get("JOI_V15_LOCAL_FILES_ONLY"))
    print("JOI_V15_PERSISTENT_WORKER:", os.environ.get("JOI_V15_PERSISTENT_WORKER"))

setup_env_for_current_server()


# =============================================================================
# 5. Cloud advisor API key 확인
# =============================================================================

if not os.environ.get("OPENAI_API_KEY"):
    print("[WARN] OPENAI_API_KEY is not set. cloud_advisor runs will fail unless set.")
else:
    print("[OK] OPENAI_API_KEY is set.")

## API key 입력 셀

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

## Cell 2. run_ga_all_categories wrapper 정의


In [ ]:
# =============================================================================
# run_ga_search.py CLI wrapper
# =============================================================================

def _get_run_ga_help_text():
    try:
        proc = subprocess.run(
            [sys.executable, str(SCRIPT), "--help"],
            cwd=str(REPO),
            env=os.environ.copy(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=60,
        )
        return proc.stdout or ""
    except Exception as e:
        print("[WARN] failed to inspect --help:", repr(e))
        return ""

_RUN_GA_HELP_TEXT = _get_run_ga_help_text()

def _has_flag(flag: str) -> bool:
    if not _RUN_GA_HELP_TEXT:
        return True
    return flag in _RUN_GA_HELP_TEXT

def _append_optional_flag(cmd, flag, value=None):
    if _has_flag(flag):
        cmd.append(flag)
        if value is not None:
            cmd.append(str(value))
    else:
        print(f"[SKIP unsupported flag] {flag}")


def run_ga_all_categories(
    model_key: str,
    categories=range(1, 9),
    limit_per_category=3,
    sample_size=24,
    validation_size=24,
    population=5,
    gens=10,
    target_detpass=90,
    base_prefix="ga_final",
    use_advisor=False,
    full_run=True,
    progress="verbose",
    timeout_sec=600,
    retries=0,
    idle_timeout_sec=2400,
    total_timeout_sec=32 * 3600,

    advisor_trigger_mode="always",
    advisor_min_population_for_child=4,
    advisor_force_child_quota=True,
    use_mock_advisor=False,

    launcher_python=None,
):
    if launcher_python is None:
        launcher_python = sys.executable

    # resolve() 금지. symlink가 conda 원본으로 풀리는 문제 방지.
    launcher_python = os.path.abspath(os.path.expanduser(launcher_python))

    if not Path(launcher_python).exists():
        raise FileNotFoundError(f"launcher_python does not exist: {launcher_python}")

    if use_advisor and not use_mock_advisor:
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY is not set for real cloud advisor mode.")

    categories = tuple(categories)
    cat_text = "".join(str(c) for c in categories)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    mode = "cloud_advisor" if use_advisor else "cloudless"
    mock_tag = "_mock" if use_advisor and use_mock_advisor else ""
    safe_model_key = model_key.replace("/", "_").replace(":", "_")

    run_name = (
        f"{base_prefix}_{mode}{mock_tag}_cat{cat_text}"
        f"_lpc{limit_per_category}_pop{population}_gens{gens}"
        f"_{safe_model_key}_{ts}"
    )

    out_dir = RESULTS_ROOT / run_name / "ga_output"

    cmd = [
        launcher_python,
        "-u",
        str(SCRIPT),

        "--profile", "version0_15",
        "--model-key", model_key,
        "--target-detpass", str(target_detpass),

        "--population", str(population),
        "--gens", str(gens),
        "--min-generations", str(gens),
        "--max-generations", str(gens),

        "--sample-size", str(sample_size),
        "--validation-size", str(validation_size),
        "--cheap-eval-limit", "2",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--feedback-guided-mutation",

        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",

        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--plateau-window", "1",
        "--disruptive-max-attempts", "1",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",

        "--progress", str(progress),
        "--timeout-sec", str(timeout_sec),
        "--retries", str(retries),
        "--limit-per-category", str(limit_per_category),
        "--output-root", str(out_dir),
    ]

    _append_optional_flag(cmd, "--idle-timeout-sec", idle_timeout_sec)
    _append_optional_flag(cmd, "--total-timeout-sec", total_timeout_sec)

    if full_run:
        _append_optional_flag(cmd, "--full-run")

    for c in categories:
        cmd += ["--category", str(c)]

    if use_advisor:
        cmd += [
            "--llm-mutation-advisor",
            "--advisor-model-key", "gpt41_mini",
            "--advisor-trigger-mode", advisor_trigger_mode,
            "--advisor-min-population-for-child", str(advisor_min_population_for_child),
        ]

        if advisor_force_child_quota:
            cmd += ["--advisor-force-child-quota"]

        if use_mock_advisor:
            cmd += ["--llm-mode", "mock"]
    else:
        cmd += ["--advisor-trigger-mode", "off"]

    run_env = os.environ.copy()
    run_env["PYTHONUNBUFFERED"] = "1"
    run_env["JOI_V15_WORKER_PYTHON"] = launcher_python

    # 매우 중요: persistent worker를 끄지 않는다.
    if run_env.get("JOI_V15_PERSISTENT_WORKER") == "false":
        run_env.pop("JOI_V15_PERSISTENT_WORKER", None)

    debug_log = f"/tmp/joi_v15_worker_debug_{safe_model_key}_{mode}_{ts}.log"
    run_env["JOI_V15_DEBUG_WORKER"] = "1"
    run_env["JOI_V15_DEBUG_LOG"] = debug_log

    print("=" * 120)
    print(f"RUN: {model_key} / {mode}{mock_tag}")
    print("OUTPUT:", out_dir)
    print("LAUNCHER_PYTHON:", launcher_python)
    print("JOI_V15_WORKER_PYTHON:", run_env.get("JOI_V15_WORKER_PYTHON"))
    print("JOI_V15_LOCAL_MODEL_BASE_DIR:", run_env.get("JOI_V15_LOCAL_MODEL_BASE_DIR"))
    print("JOI_V15_LOCAL_DEVICE:", run_env.get("JOI_V15_LOCAL_DEVICE"))
    print("JOI_V15_LOCAL_FILES_ONLY:", run_env.get("JOI_V15_LOCAL_FILES_ONLY"))
    print("JOI_V15_PERSISTENT_WORKER:", run_env.get("JOI_V15_PERSISTENT_WORKER"))
    print("DEBUG_LOG:", debug_log)
    print("COMMAND:")
    print(" ".join(cmd))
    print("=" * 120)

    proc = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert proc.stdout is not None

    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()

    print("\nRETURN CODE:", rc)
    print("OUTPUT:", out_dir)
    print("DEBUG_LOG:", debug_log)

    if rc != 0:
        raise RuntimeError(f"run_ga_all_categories failed with return code {rc}")

    return out_dir

## Cell 3. 3개 모델 × 2조건 자동 실행

### 본격 6개 run 전에 반드시 1회만 테스트

In [ ]:
sanity_out = run_ga_all_categories(
    model_key="qwen25_coder_7b",
    categories=range(1, 3),
    limit_per_category=1,
    sample_size=2,
    validation_size=2,
    population=2,
    gens=1,
    target_detpass=90,
    base_prefix=f"sanity_{SERVER_PRESET}",
    use_advisor=False,
    full_run=True,
    progress="verbose",
    timeout_sec=600,
    retries=0,
)

sanity_out

In [ ]:
ga_runs_fair = {}

for label, model_key in MODEL_LIST:
    for mode_name, use_advisor in RUN_MODES:
        run_key = f"{label}_{mode_name}"

        print("\n" + "#" * 120)
        print(f"START RUN: {run_key} / {model_key}")
        print("#" * 120)

        try:
            out_dir = run_ga_all_categories(
                model_key=model_key,
                categories=COMMON_GA_CONFIG["categories"],
                limit_per_category=COMMON_GA_CONFIG["limit_per_category"],
                sample_size=COMMON_GA_CONFIG["sample_size"],
                validation_size=COMMON_GA_CONFIG["validation_size"],
                population=COMMON_GA_CONFIG["population"],
                gens=COMMON_GA_CONFIG["gens"],
                target_detpass=COMMON_GA_CONFIG["target_detpass"],
                base_prefix=f"ga_{SERVER_PRESET}",
                use_advisor=use_advisor,
                full_run=COMMON_GA_CONFIG["full_run"],
                progress=COMMON_GA_CONFIG["progress"],
                timeout_sec=COMMON_GA_CONFIG["timeout_sec"],
                retries=COMMON_GA_CONFIG["retries"],
                idle_timeout_sec=COMMON_GA_CONFIG["idle_timeout_sec"],
                total_timeout_sec=COMMON_GA_CONFIG["total_timeout_sec"],

                advisor_trigger_mode="always",
                advisor_min_population_for_child=4,
                advisor_force_child_quota=True,
                use_mock_advisor=False,
            )

            ga_runs_fair[run_key] = out_dir
            print(f"[PASS] {run_key}: {out_dir}")

        except Exception as e:
            print(f"[FAIL] {run_key}: {type(e).__name__}: {e}")
            traceback.print_exc()
            ga_runs_fair[run_key] = None

        time.sleep(5)


valid_ga_runs_fair = {
    key: path
    for key, path in ga_runs_fair.items()
    if path is not None
}

valid_ga_runs_fair

## Cell 4. 결과 요약표 생성


In [ ]:
def load_json_safe(path: Path, default=None):
    if default is None:
        default = {}
    try:
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    except Exception:
        pass
    return default


def read_csv_safe(path: Path):
    try:
        if path.exists() and path.stat().st_size > 0:
            return pd.read_csv(path)
    except Exception:
        pass
    return pd.DataFrame()


def summarize_ga_run(run_key: str, out_dir):
    out_dir = Path(out_dir)

    summary = load_json_safe(out_dir / "ga_summary.json", default={})
    progress_df = read_csv_safe(out_dir / "ga_generation_progress.csv")
    diag_df = read_csv_safe(out_dir / "ga_population_diagnostics.csv")

    last_progress = {}
    if not progress_df.empty:
        last_progress = progress_df.tail(1).iloc[0].to_dict()

    failure_histogram = None
    if not diag_df.empty and "failure_histogram" in diag_df.columns:
        failure_histogram = diag_df.tail(1)["failure_histogram"].iloc[0]

    label, mode = run_key.split("_", 1)

    return {
        "server_preset": SERVER_PRESET,
        "run_key": run_key,
        "label": label,
        "mode": mode,
        "out_dir": str(out_dir),

        "target_detpass": COMMON_GA_CONFIG["target_detpass"],
        "categories": ",".join(map(str, COMMON_GA_CONFIG["categories"])),
        "limit_per_category": COMMON_GA_CONFIG["limit_per_category"],
        "sample_size": COMMON_GA_CONFIG["sample_size"],
        "validation_size": COMMON_GA_CONFIG["validation_size"],
        "population": COMMON_GA_CONFIG["population"],
        "gens": COMMON_GA_CONFIG["gens"],

        "stage": summary.get("stage"),
        "stop_reason": summary.get("stop_reason"),
        "best_genome_id": summary.get("best_genome_id"),
        "best_generation": summary.get("best_generation"),

        "best_DETPass": summary.get("best_DETPass"),
        "best_avg_DET": summary.get("best_avg_DET"),
        "accepted_best_DETPass": summary.get("accepted_best_DETPass"),
        "accepted_best_avg_DET": summary.get("accepted_best_avg_DET"),
        "accepted_best_tokens": summary.get("accepted_best_tokens"),

        "compact_best_eligible": summary.get("compact_best_eligible"),
        "compact_best_DETPass": summary.get("compact_best_DETPass"),
        "compact_best_tokens": summary.get("compact_best_tokens"),

        "advisor_status": summary.get("advisor_status"),
        "advisor_used": summary.get("advisor_used"),
        "advisor_proposals_generated": summary.get("advisor_proposals_generated"),
        "advisor_proposals_accepted_applied": summary.get("advisor_proposals_accepted_applied"),
        "advisor_proposals_rejected": summary.get("advisor_proposals_rejected"),
        "advisor_children_scheduled": summary.get("advisor_children_scheduled"),

        "cloudless_mutation_used": summary.get("cloudless_mutation_used"),
        "pareto_archive_size": summary.get("pareto_archive_size"),

        "last_progress_fitness": last_progress.get("fitness"),
        "last_progress_avg_det_score": last_progress.get("avg_det_score"),
        "last_progress_det_pass_rate": last_progress.get("det_pass_rate"),
        "last_progress_advisor_triggered": last_progress.get("advisor_triggered"),

        "failure_histogram": failure_histogram,

        "summary_exists": (out_dir / "ga_summary.json").exists(),
        "progress_exists": (out_dir / "ga_generation_progress.csv").exists(),
    }


summary_rows = []

for run_key, out_dir in ga_runs_fair.items():
    if out_dir is None:
        label, mode = run_key.split("_", 1)
        summary_rows.append({
            "server_preset": SERVER_PRESET,
            "run_key": run_key,
            "label": label,
            "mode": mode,
            "run_status": "failed",
            "out_dir": None,
        })
    else:
        row = summarize_ga_run(run_key, out_dir)
        row["run_status"] = "success"
        summary_rows.append(row)

fair_summary_df = pd.DataFrame(summary_rows)

summary_csv = COMPARISON_ROOT / f"{SERVER_PRESET}_fair_summary.csv"
fair_summary_df.to_csv(summary_csv, index=False)

print("summary_csv:", summary_csv)
display(fair_summary_df)

## Cell 5. cloudless vs cloud-advisor delta 표 생성


In [ ]:
def make_delta_table(fair_summary_df):
    df = fair_summary_df.copy()
    df = df[df["run_status"] == "success"].copy()

    rows = []

    for label in ["7B", "8B", "14B"]:
        c = df[(df["label"] == label) & (df["mode"] == "cloudless")]
        a = df[(df["label"] == label) & (df["mode"] == "cloud_advisor")]

        if c.empty or a.empty:
            rows.append({
                "server_preset": SERVER_PRESET,
                "label": label,
                "status": "missing_pair",
            })
            continue

        c = c.iloc[0]
        a = a.iloc[0]

        row = {
            "server_preset": SERVER_PRESET,
            "label": label,
            "status": "ok",

            "cloudless_best_DETPass": c.get("best_DETPass"),
            "advisor_best_DETPass": a.get("best_DETPass"),
            "delta_best_DETPass": None,

            "cloudless_best_avg_DET": c.get("best_avg_DET"),
            "advisor_best_avg_DET": a.get("best_avg_DET"),
            "delta_best_avg_DET": None,

            "cloudless_tokens": c.get("accepted_best_tokens"),
            "advisor_tokens": a.get("accepted_best_tokens"),
            "delta_tokens": None,

            "cloudless_best_generation": c.get("best_generation"),
            "advisor_best_generation": a.get("best_generation"),

            "advisor_used": a.get("advisor_used"),
            "advisor_proposals_generated": a.get("advisor_proposals_generated"),
            "advisor_proposals_accepted_applied": a.get("advisor_proposals_accepted_applied"),
            "advisor_children_scheduled": a.get("advisor_children_scheduled"),

            "cloudless_out_dir": c.get("out_dir"),
            "advisor_out_dir": a.get("out_dir"),
        }

        for out_key, adv_key, base_key in [
            ("delta_best_DETPass", "advisor_best_DETPass", "cloudless_best_DETPass"),
            ("delta_best_avg_DET", "advisor_best_avg_DET", "cloudless_best_avg_DET"),
            ("delta_tokens", "advisor_tokens", "cloudless_tokens"),
        ]:
            try:
                row[out_key] = row[adv_key] - row[base_key]
            except Exception:
                row[out_key] = None

        rows.append(row)

    delta_df = pd.DataFrame(rows)

    delta_csv = COMPARISON_ROOT / f"{SERVER_PRESET}_cloudless_vs_advisor_delta.csv"
    delta_df.to_csv(delta_csv, index=False)

    print("delta_csv:", delta_csv)
    display(delta_df)

    return delta_df


fair_delta_df = make_delta_table(fair_summary_df)

## Cell 6. advisor activity 확인

In [ ]:
def inspect_advisor_activity(ga_runs_fair):
    rows = []

    for run_key, out_dir in ga_runs_fair.items():
        if out_dir is None:
            continue

        out_dir = Path(out_dir)

        advisor_feedback = out_dir / "advisor_feedback_batches.jsonl"
        advisor_proposals = out_dir / "advisor_mutation_proposals.jsonl"
        advisor_summary = out_dir / "advisor_mutation_summary.csv"

        rows.append({
            "server_preset": SERVER_PRESET,
            "run_key": run_key,
            "out_dir": str(out_dir),
            "advisor_feedback_exists": advisor_feedback.exists(),
            "advisor_feedback_bytes": advisor_feedback.stat().st_size if advisor_feedback.exists() else 0,
            "advisor_proposals_exists": advisor_proposals.exists(),
            "advisor_proposals_bytes": advisor_proposals.stat().st_size if advisor_proposals.exists() else 0,
            "advisor_summary_exists": advisor_summary.exists(),
            "advisor_summary_bytes": advisor_summary.stat().st_size if advisor_summary.exists() else 0,
        })

    advisor_df = pd.DataFrame(rows)

    advisor_csv = COMPARISON_ROOT / f"{SERVER_PRESET}_advisor_activity.csv"
    advisor_df.to_csv(advisor_csv, index=False)

    print("advisor_csv:", advisor_csv)
    display(advisor_df)

    return advisor_df


advisor_activity_df = inspect_advisor_activity(ga_runs_fair)